# CNN transfer: Sensorium pretrain → Mouse (frozen backbone, new readout)

- **Phase 1:** Train on Sensorium with **random 70/30** split (`split_strategy=random`), **shifter off**, weights under `transfer_learning_weights/`.
- **Phase 2:** Load frozen CNN backbone, train **readout** on Mouse (same metrics / pickle layout as `train_cnn_shifter_mouse_vs_sensorium`).
- **Baseline:** Same Mouse setup, **random init** (no transfer).

Configure paths in the next cell, then run all.

In [1]:
import os
import pickle
import random
import numpy as np
import torch

from torch.utils.data import Subset, DataLoader, ConcatDataset

from mouse_model.sensorium_dataset import SensoriumDataset
from mouse_model.data_utils_new import MouseDatasetSegNewBehav
from mouse_model.cnn_predictor_transfer import (
    PredictorTransfer,
    load_encoder_backbone_from_checkpoint,
    freeze_encoder_backbone,
)
from mouse_model.transfer_cnn_training import train_cnn_transfer

# --- User paths ---
SENSORIUM_ROOT = os.path.expanduser(
    "/home/herbelinluke/Downloads/dynamic29515-10-12-Video-9b4f6a1a067fe51e15306b9628efea20"
)
MOUSE_FILE_ID = "070921_J553RT"
MOUSE_SEGMENT_NUM = 10
MOUSE_VID_TYPE = "vid_mean"

WEIGHT_DIR = os.path.join(os.getcwd(), "transfer_learning_weights")
os.makedirs(WEIGHT_DIR, exist_ok=True)

# Training hyperparameters (match your usual CNN runs)
SEED = 0
EPOCHS = 100
BATCH_SIZE = 256
LEARNING_RATE = 1e-4
SEQ_LEN = 1
VID_FRAME_SENSORIUM = "per_frame"  # align with mouse vid_mean (single frame per sample)

# Sensorium random split (70/30)
SPLIT_STRATEGY = "random"
TRAIN_RATIO = 0.7

USE_SHIFTER = False


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

print("device:", device)


device: cuda


In [3]:
def _get_or_create_split(full_ds, split_tag, train_ratio, seed, sensorium_root):
    import pickle as _p
    ratio_str = str(int(train_ratio * 100))
    split_path = os.path.join(
        sensorium_root, "meta", "trials",
        f"split_{ratio_str}_{100 - int(train_ratio * 100)}_{split_tag}.pkl",
    )
    n = len(full_ds)
    if os.path.isfile(split_path):
        with open(split_path, "rb") as f:
            sp = _p.load(f)
        if len(sp["train_indices"]) + len(sp["val_indices"]) == n:
            print(f"Loaded split from {split_path}")
            return sp["train_indices"], sp["val_indices"]
    n_train = int(n * train_ratio)
    indices = np.random.RandomState(seed).permutation(n)
    train_indices = indices[:n_train].tolist()
    val_indices = indices[n_train:].tolist()
    os.makedirs(os.path.dirname(split_path), exist_ok=True)
    with open(split_path, "wb") as f:
        _p.dump(
            {"train_indices": train_indices, "val_indices": val_indices,
             "seed": seed, "train_ratio": train_ratio, "tag": split_tag},
            f,
        )
    print(f"Saved split to {split_path}")
    return train_indices, val_indices


def load_sensorium_train_val():
    full_ds = SensoriumDataset(
        root_dir=SENSORIUM_ROOT,
        data_split="all",
        seq_len=SEQ_LEN,
        vid_frame=VID_FRAME_SENSORIUM,
        standardize_responses=True,
    )
    tr_idx, va_idx = _get_or_create_split(full_ds, "all", TRAIN_RATIO, SEED, SENSORIUM_ROOT)
    train_ds = Subset(full_ds, tr_idx)
    val_ds = Subset(full_ds, va_idx)
    n_neurons = full_ds.num_neurons
    print(f"Sensorium train={len(train_ds)} val={len(val_ds)} num_neurons={n_neurons}")
    return train_ds, val_ds, n_neurons


def load_mouse_train_val(max_train_samples=None):
    ds_list = [
        MouseDatasetSegNewBehav(
            file_id=MOUSE_FILE_ID,
            segment_num=MOUSE_SEGMENT_NUM,
            seg_idx=i,
            data_split="train",
            vid_type=MOUSE_VID_TYPE,
            seq_len=SEQ_LEN,
            predict_offset=1,
        )
        for i in range(MOUSE_SEGMENT_NUM)
    ]
    train_ds, val_ds = [], []
    for ds in ds_list:
        train_ratio_m = 0.8
        train_ds_len = int(len(ds) * train_ratio_m)
        train_ds.append(Subset(ds, np.arange(0, train_ds_len)))
        val_ds.append(Subset(ds, np.arange(train_ds_len, len(ds))))
    train_ds = ConcatDataset(train_ds)
    val_ds = ConcatDataset(val_ds)
    if max_train_samples is not None:
        n_total = len(train_ds)
        n_use = min(max_train_samples, n_total)
        perm = np.random.RandomState(SEED).permutation(n_total)[:n_use]
        train_ds = Subset(train_ds, perm)
        print(f"max_train_samples={max_train_samples} -> train={n_use} val={len(val_ds)}")
    else:
        print(f"Mouse train={len(train_ds)} val={len(val_ds)}")
    _probe = MouseDatasetSegNewBehav(
        file_id=MOUSE_FILE_ID,
        segment_num=MOUSE_SEGMENT_NUM,
        seg_idx=0,
        data_split="train",
        vid_type=MOUSE_VID_TYPE,
        seq_len=SEQ_LEN,
        predict_offset=1,
    )
    num_neurons = _probe.nsp.shape[1]
    return train_ds, val_ds, num_neurons


In [4]:
# Phase 1 — pretrain on Sensorium (full readout, all encoder weights trainable)
train_s, val_s, n_neurons_s = load_sensorium_train_val()

pretrain_train_path = os.path.join(WEIGHT_DIR, "sensorium_pretrain_train.pth")
pretrain_val_path = os.path.join(WEIGHT_DIR, "sensorium_pretrain_val.pth")

model_s = PredictorTransfer(num_neurons=n_neurons_s, use_shifter=USE_SHIFTER).to(device)

(train_loss_s, val_loss_s, val_cor_s, val_r2_s, val_mse_s, val_pl_s, val_bps_s, val_ev_s,
 cor_pn_s, r2_pn_s, ev_pn_s, n_valid_s) = train_cnn_transfer(
    model_s,
    device,
    train_s,
    val_s,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    seed=SEED,
    best_train_path=pretrain_train_path,
    best_val_path=pretrain_val_path,
    num_workers=8,
    freeze_encoder_backbone=False,
)

print("Phase 1 best checkpoints:", pretrain_train_path, pretrain_val_path)


Loaded split from /home/herbelinluke/Downloads/dynamic29515-10-12-Video-9b4f6a1a067fe51e15306b9628efea20/meta/trials/split_70_30_all.pkl
Sensorium train=17917 val=7679 num_neurons=7863
Start epoch 0


/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimod

Epoch 0 train loss: 0.5784


/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimod

Epoch 0 val loss: 0.4467 | corr: 0.1031 R2: -0.0899 MSE: 0.4758 EV: 0.0060 | valid neurons: 7863 / 7863
End epoch 0
Start epoch 1


...

Epoch 98 val loss: 0.2912 | corr: 0.2545 R2: 0.0660 MSE: 0.4101 EV: 0.0666 | valid neurons: 7863 / 7863
End epoch 98
Start epoch 99


/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimod

Epoch 99 train loss: 0.2851


/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/mouse_model/sensorium_dataset.py:72: RuntimeWarning: Mean of empty slice
  out[f] = np.nanmean(behav_2d[:, start:end], axis=-1)
/home/herbelinluke/Documents/bioV/2023-Xu-Multimod

Epoch 99 val loss: 0.2913 | corr: 0.2552 R2: 0.0659 MSE: 0.4101 EV: 0.0667 | valid neurons: 7863 / 7863
End epoch 99
Phase 1 best checkpoints: /home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/transfer_learning_weights/sensorium_pretrain_train.pth /home/herbelinluke/Documents/bioV/2023-Xu-Multimodal-Mouse-V1/transfer_learning_weights/sensorium_pretrain_val.pth


In [5]:
def save_mouse_metrics_pkl(
    *,
    fname_suffix,
    train_loss_list,
    val_loss_list,
    val_cor_list,
    val_r2_list,
    val_mse_list,
    val_poisson_loss_list,
    val_bits_per_spike_list,
    val_explained_var_list,
    cor_per_neuron_per_epoch,
    r2_per_neuron_per_epoch,
    ev_per_neuron_per_epoch,
    n_valid_per_epoch,
    extra_meta,
    max_train_samples=None,
):
    base_meta = {
        "model_name": "cnn",
        "file_id": MOUSE_FILE_ID,
        "vid_type": MOUSE_VID_TYPE,
        "shifter": USE_SHIFTER,
        "dataset": "mouse",
        "max_train_samples": max_train_samples,
        "segment_num": MOUSE_SEGMENT_NUM,
        "train_loss_list": train_loss_list,
        "val_loss_list": val_loss_list,
        **extra_meta,
    }
    fname = (
        f"epoch_vs_score_cnn_mouse_{MOUSE_FILE_ID}_{MOUSE_VID_TYPE}_shifter_{USE_SHIFTER}_"
        f"{fname_suffix}.pkl"
    )
    score_dirs = [
        ("epoch_vs_score_data", "val_cor_list", val_cor_list),
        ("epoch_vs_correlation_data", "val_cor_list", val_cor_list),
        ("epoch_vs_r2_data", "val_r2_list", val_r2_list),
        ("epoch_vs_mse_data", "val_mse_list", val_mse_list),
        ("epoch_vs_poisson_loss_data", "val_poisson_loss_list", val_poisson_loss_list),
        ("epoch_vs_bits_per_spike_data", "val_bits_per_spike_list", val_bits_per_spike_list),
        ("epoch_vs_explained_variance_data", "val_explained_var_list", val_explained_var_list),
    ]
    for dir_name, score_key, score_list in score_dirs:
        os.makedirs(dir_name, exist_ok=True)
        path = os.path.join(dir_name, fname)
        with open(path, "wb") as f:
            pickle.dump({**base_meta, score_key: score_list}, f)
        print("Saved", path)
    pnd = "epoch_vs_per_neuron_data"
    os.makedirs(pnd, exist_ok=True)
    pn_fname = f"per_neuron_cnn_mouse_{MOUSE_FILE_ID}_{MOUSE_VID_TYPE}_shifter_{USE_SHIFTER}_{fname_suffix}.pkl"
    pn_path = os.path.join(pnd, pn_fname)
    with open(pn_path, "wb") as f:
        pickle.dump({
            **base_meta,
            "cor_per_neuron_per_epoch": cor_per_neuron_per_epoch,
            "r2_per_neuron_per_epoch": r2_per_neuron_per_epoch,
            "ev_per_neuron_per_epoch": ev_per_neuron_per_epoch,
            "n_valid_per_epoch": n_valid_per_epoch,
        }, f)
    print("Saved", pn_path)


In [6]:
# Phase 2 — Mouse with frozen backbone initialized from Sensorium val checkpoint
train_m, val_m, n_neurons_m = load_mouse_train_val(max_train_samples=None)

try:
    sd = torch.load(pretrain_val_path, map_location=device, weights_only=False)
except TypeError:
    sd = torch.load(pretrain_val_path, map_location=device)
model_t = PredictorTransfer(num_neurons=n_neurons_m, use_shifter=USE_SHIFTER).to(device)
load_encoder_backbone_from_checkpoint(model_t, sd)
freeze_encoder_backbone(model_t)

transfer_train_path = os.path.join(WEIGHT_DIR, "mouse_transfer_train.pth")
transfer_val_path = os.path.join(WEIGHT_DIR, "mouse_transfer_val.pth")

extra_transfer = {
    "pretrained_source": "sensorium",
    "pretrained_checkpoint": pretrain_val_path,
    "freeze_policy": "encoder_backbone",
    "training_mode": "transfer_frozen_backbone",
}

(tl_t, vl_t, vc_t, vr_t, vm_t, vp_t, vb_t, ve_t, cpn_t, rpn_t, evn_t, nv_t) = train_cnn_transfer(
    model_t,
    device,
    train_m,
    val_m,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    seed=SEED,
    best_train_path=transfer_train_path,
    best_val_path=transfer_val_path,
    num_workers=8,
    freeze_encoder_backbone=True,
)

save_mouse_metrics_pkl(
    fname_suffix="transfer_from_sensorium_frozen_backbone_maxsamp_full",
    train_loss_list=tl_t,
    val_loss_list=vl_t,
    val_cor_list=vc_t,
    val_r2_list=vr_t,
    val_mse_list=vm_t,
    val_poisson_loss_list=vp_t,
    val_bits_per_spike_list=vb_t,
    val_explained_var_list=ve_t,
    cor_per_neuron_per_epoch=cpn_t,
    r2_per_neuron_per_epoch=rpn_t,
    ev_per_neuron_per_epoch=evn_t,
    n_valid_per_epoch=nv_t,
    extra_meta=extra_transfer,
    max_train_samples=None,
)


Mouse train=30120 val=7540
Start epoch 0
Epoch 0 train loss: 0.7728
Epoch 0 val loss: 0.7182 | corr: 0.0260 R2: -0.0799 MSE: 0.8304 EV: -0.0062 | valid neurons: 68 / 68
End epoch 0
Start epoch 1
Epoch 1 train loss: 0.6842
Epoch 1 val loss: 0.6818 | corr: 0.0640 R2: -0.0206 MSE: 0.7543 EV: 0.0036 | valid neurons: 68 / 68
End epoch 1
Start epoch 2
Epoch 2 train loss: 0.6628
Epoch 2 val loss: 0.6688 | corr: 0.0936 R2: -0.0015 MSE: 0.7173 EV: 0.0110 | valid neurons: 68 / 68
End epoch 2
Start epoch 3
Epoch 3 train loss: 0.6535
Epoch 3 val loss: 0.6619 | corr: 0.1152 R2: 0.0090 MSE: 0.6968 EV: 0.0167 | valid neurons: 68 / 68
End epoch 3
Start epoch 4
Epoch 4 train loss: 0.6479
Epoch 4 val loss: 0.6575 | corr: 0.1311 R2: 0.0159 MSE: 0.6847 EV: 0.0213 | valid neurons: 68 / 68
End epoch 4
Start epoch 5
Epoch 5 train loss: 0.6440
Epoch 5 val loss: 0.6544 | corr: 0.1428 R2: 0.0210 MSE: 0.6772 EV: 0.0249 | valid neurons: 68 / 68
End epoch 5
Start epoch 6
Epoch 6 train loss: 0.6410
Epoch 6 val loss

In [7]:
# Baseline — Mouse from scratch (same data as Phase 2)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

model_b = PredictorTransfer(num_neurons=n_neurons_m, use_shifter=USE_SHIFTER).to(device)

scratch_train_path = os.path.join(WEIGHT_DIR, "mouse_scratch_train.pth")
scratch_val_path = os.path.join(WEIGHT_DIR, "mouse_scratch_val.pth")

extra_scratch = {
    "pretrained_source": None,
    "pretrained_checkpoint": None,
    "freeze_policy": None,
    "training_mode": "from_scratch",
}

(tl_b, vl_b, vc_b, vr_b, vm_b, vp_b, vb_b, ve_b, cpn_b, rpn_b, evn_b, nv_b) = train_cnn_transfer(
    model_b,
    device,
    train_m,
    val_m,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    seed=SEED,
    best_train_path=scratch_train_path,
    best_val_path=scratch_val_path,
    num_workers=8,
    freeze_encoder_backbone=False,
)

save_mouse_metrics_pkl(
    fname_suffix="baseline_scratch_maxsamp_full",
    train_loss_list=tl_b,
    val_loss_list=vl_b,
    val_cor_list=vc_b,
    val_r2_list=vr_b,
    val_mse_list=vm_b,
    val_poisson_loss_list=vp_b,
    val_bits_per_spike_list=vb_b,
    val_explained_var_list=ve_b,
    cor_per_neuron_per_epoch=cpn_b,
    r2_per_neuron_per_epoch=rpn_b,
    ev_per_neuron_per_epoch=evn_b,
    n_valid_per_epoch=nv_b,
    extra_meta=extra_scratch,
    max_train_samples=None,
)


Start epoch 0
Epoch 0 train loss: 0.7463
Epoch 0 val loss: 0.6735 | corr: 0.0709 R2: -0.0090 MSE: 0.7258 EV: 0.0048 | valid neurons: 68 / 68
End epoch 0
Start epoch 1
Epoch 1 train loss: 0.6747
Epoch 1 val loss: 0.6587 | corr: 0.1196 R2: 0.0138 MSE: 0.6856 EV: 0.0186 | valid neurons: 68 / 68
End epoch 1
Start epoch 2
Epoch 2 train loss: 0.6619
Epoch 2 val loss: 0.6516 | corr: 0.1487 R2: 0.0253 MSE: 0.6698 EV: 0.0283 | valid neurons: 68 / 68
End epoch 2
Start epoch 3
Epoch 3 train loss: 0.6526
Epoch 3 val loss: 0.6463 | corr: 0.1721 R2: 0.0341 MSE: 0.6599 EV: 0.0369 | valid neurons: 68 / 68
End epoch 3
Start epoch 4
Epoch 4 train loss: 0.6451
Epoch 4 val loss: 0.6417 | corr: 0.1891 R2: 0.0415 MSE: 0.6549 EV: 0.0435 | valid neurons: 68 / 68
End epoch 4
Start epoch 5
Epoch 5 train loss: 0.6394
Epoch 5 val loss: 0.6397 | corr: 0.1983 R2: 0.0453 MSE: 0.6520 EV: 0.0478 | valid neurons: 68 / 68
End epoch 5
Start epoch 6
Epoch 6 train loss: 0.6354
Epoch 6 val loss: 0.6365 | corr: 0.2096 R2: 0.